# Continue training the 96×96 STL-10 WGAN-GP for 10 more epochs

Before running: (1) create a **private Kaggle dataset** containing the downloaded `generator.pth` and `critic.pth` checkpoint files from your previous run; (2) in this notebook use **Add Input** to attach that checkpoint dataset and [STL-10 by jessicali9530](https://www.kaggle.com/datasets/jessicali9530/stl10); (3) select **Settings → Accelerator → GPU T4 x2** (or the available two-GPU option); and (4) run all cells. The checkpoint dataset must contain both files in the same folder. The notebook auto-detects that folder under `/kaggle/input` and reads the `unlabeled_images` and `train_images` splits, excluding `test_images`.

This continues from the epoch-5 weights for **10 additional epochs**, numbered 6–15. The previous run saved model weights but not Adam optimizer state, so the weights continue training while the Adam optimizers start fresh. New checkpoints and one sample grid per epoch are written to `/kaggle/working/results_stl10_continued`; download them from the Kaggle **Output** panel after completion. If auto-detection finds zero or multiple checkpoint pairs, set `CHECKPOINT_DIR` explicitly in the configuration cell.

In [ ]:
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid, save_image
from tqdm.auto import tqdm

SEED = 42
IMAGE_SIZE = 96
Z_DIM = 100
BASE_CHANNELS = 64
BATCH_SIZE = 128
EPOCHS = 10
START_EPOCH = 5
CHECKPOINT_DIR = None  # Or set Path('/kaggle/input/<your-checkpoint-dataset>/<folder-with-both-files>')
N_CRITIC = 5
LAMBDA_GP = 10.0
LEARNING_RATE = 1e-4
NUM_WORKERS = 2
OUTPUT_DIR = Path('/kaggle/working/results_stl10_continued')
SAMPLE_COUNT = 16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pin_memory = device.type == 'cuda'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'samples').mkdir(exist_ok=True)
(OUTPUT_DIR / 'checkpoints').mkdir(exist_ok=True)
print(f'Device: {device}')
if device.type != 'cuda':
    print('GPU is not enabled. In Kaggle, set Settings → Accelerator → GPU for practical training speed.')

In [ ]:
# Kaggle mounts attached datasets beneath /kaggle/input. Find image files
# inside the expected split folders, regardless of the dataset's subfolder name.
input_root = Path('/kaggle/input')
if not input_root.is_dir():
    raise FileNotFoundError('Kaggle input directory /kaggle/input is missing.')

image_extensions = {'.png', '.jpg', '.jpeg'}
all_images = sorted(
    path for path in input_root.rglob('*')
    if path.is_file() and path.suffix.lower() in image_extensions
)
preferred_images = [
    path for path in all_images
    if any(part.lower() in {'unlabeled_images', 'train_images'} for part in path.parts)
]
if preferred_images:
    image_paths = preferred_images
else:
    # Fallback for alternate layouts: include image files except test-split files.
    image_paths = [path for path in all_images if not any('test' in part.lower() for part in path.parts)]

if not image_paths:
    raise FileNotFoundError(
        'No STL-10 PNG/JPEG images found under /kaggle/input. '
        'Use Add Input to attach jessicali9530/stl10, then rerun this cell.'
    )
print(f'Using {len(image_paths):,} training images; test split is excluded.')
print('Example image:', image_paths[0])

class STL10ImageDataset(Dataset):
    def __init__(self, paths, transform=None):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = image.convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image

image_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
dataset = STL10ImageDataset(image_paths, image_transform)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=True,
    persistent_workers=NUM_WORKERS > 0,
)
if len(loader) == 0:
    raise ValueError(f'No full batches: dataset has {len(dataset)} images but batch size is {BATCH_SIZE}.')
print(f'{len(loader):,} batches per epoch at batch size {BATCH_SIZE}.')

In [ ]:
def gen_block(in_channels, out_channels, kernel=4, stride=2, padding=1):
    return [
        nn.ConvTranspose2d(in_channels, out_channels, kernel, stride, padding, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    ]

def critic_block(in_channels, out_channels):
    return [
        nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False),
        nn.InstanceNorm2d(out_channels, affine=True),
        nn.LeakyReLU(0.2, inplace=True),
    ]

class Generator(nn.Module):
    """96x96 generator: latent vector → 3x96x96 RGB image."""
    def __init__(self, z_dim=Z_DIM, base_ch=BASE_CHANNELS):
        super().__init__()
        ch = base_ch
        layers = [
            *gen_block(z_dim, ch * 8, kernel=3, stride=1, padding=0),  # 1 -> 3
            *gen_block(ch * 8, ch * 4),                               # 3 -> 6
            *gen_block(ch * 4, ch * 4),                               # 6 -> 12
            *gen_block(ch * 4, ch * 2),                               # 12 -> 24
            *gen_block(ch * 2, ch * 2),                               # 24 -> 48
            *gen_block(ch * 2, ch),                                   # 48 -> 96
            nn.Conv2d(ch, ch // 2, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(ch // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch // 2, 3, kernel_size=3, stride=1, padding=1, bias=False),
            nn.Tanh(),
        ]
        self.net = nn.Sequential(*layers)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.normal_(module.weight, 0.0, 0.02)
        elif isinstance(module, nn.BatchNorm2d):
            nn.init.normal_(module.weight, 1.0, 0.02)
            nn.init.zeros_(module.bias)

    def forward(self, z):
        return self.net(z.view(z.size(0), -1, 1, 1))

class Critic(nn.Module):
    """96x96 critic: RGB image → one Wasserstein score per image."""
    def __init__(self, base_ch=BASE_CHANNELS):
        super().__init__()
        ch = base_ch
        layers = [
            nn.Conv2d(3, ch // 2, kernel_size=3, stride=1, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ch // 2, ch, kernel_size=3, stride=1, padding=1, bias=False),
            nn.InstanceNorm2d(ch, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            *critic_block(ch, ch * 2),       # 96 -> 48
            *critic_block(ch * 2, ch * 4),   # 48 -> 24
            *critic_block(ch * 4, ch * 4),   # 24 -> 12
            *critic_block(ch * 4, ch * 8),   # 12 -> 6
            *critic_block(ch * 8, ch * 8),   # 6 -> 3
            nn.Conv2d(ch * 8, 1, kernel_size=3, stride=1, padding=0, bias=False),
        ]
        self.net = nn.Sequential(*layers)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Conv2d):
            nn.init.normal_(module.weight, 0.0, 0.02)

    def forward(self, images):
        return self.net(images).view(images.size(0))

netG = Generator().to(device)
netC = Critic().to(device)
if device.type == 'cuda' and torch.cuda.device_count() > 1:
    print(f'Using {torch.cuda.device_count()} GPUs with DataParallel')
    netG = nn.DataParallel(netG)
    netC = nn.DataParallel(netC)
else:
    print(f'Using one device: {device}')
print(f'Generator parameters: {sum(p.numel() for p in netG.parameters()):,}')
print(f'Critic parameters:    {sum(p.numel() for p in netC.parameters()):,}')
with torch.no_grad():
    test_output = netG(torch.randn(2, Z_DIM, device=device))
    assert tuple(test_output.shape) == (2, 3, IMAGE_SIZE, IMAGE_SIZE)
    assert tuple(netC(test_output).shape) == (2,)
print('Model shape check passed.')

In [ ]:
def gradient_penalty(critic, real, fake):
    batch = real.size(0)
    alpha = torch.rand(batch, 1, 1, 1, device=real.device)
    interpolated = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    scores = critic(interpolated)
    gradients = torch.autograd.grad(
        outputs=scores,
        inputs=interpolated,
        grad_outputs=torch.ones_like(scores),
        create_graph=True,
        retain_graph=True,
    )[0]
    gradients = gradients.reshape(batch, -1)
    return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

def save_sample_grid(epoch):
    netG.eval()
    with torch.no_grad():
        generated = (netG(fixed_noise).cpu() + 1) / 2
    netG.train()
    grid = make_grid(generated, nrow=4, padding=2)
    path = OUTPUT_DIR / 'samples' / f'epoch_{epoch:03d}.png'
    save_image(grid, path)
    return path

optimizer_G = optim.Adam(netG.parameters(), lr=LEARNING_RATE, betas=(0.0, 0.9))
optimizer_C = optim.Adam(netC.parameters(), lr=LEARNING_RATE, betas=(0.0, 0.9))
fixed_noise = torch.randn(SAMPLE_COUNT, Z_DIM, device=device)
g_losses, c_losses = [], []

for epoch in range(1, EPOCHS + 1):
    netG.train()
    netC.train()
    epoch_g, epoch_c = 0.0, 0.0
    progress = tqdm(loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=True)
    for real in progress:
        real = real.to(device, non_blocking=pin_memory)
        batch = real.size(0)

        # Critic updates
        for _ in range(N_CRITIC):
            z = torch.randn(batch, Z_DIM, device=device)
            with torch.no_grad():
                fake = netG(z)
            score_real = netC(real).mean()
            score_fake = netC(fake).mean()
            gp = gradient_penalty(netC, real, fake)
            loss_c = score_fake - score_real + LAMBDA_GP * gp
            optimizer_C.zero_grad(set_to_none=True)
            loss_c.backward()
            optimizer_C.step()

        # Generator update
        z = torch.randn(batch, Z_DIM, device=device)
        loss_g = -netC(netG(z)).mean()
        optimizer_G.zero_grad(set_to_none=True)
        loss_g.backward()
        optimizer_G.step()

        g_value, c_value = loss_g.item(), loss_c.item()
        g_losses.append(g_value)
        c_losses.append(c_value)
        epoch_g += g_value
        epoch_c += c_value
        progress.set_postfix(critic=f'{c_value:.3f}', generator=f'{g_value:.3f}')

    mean_g = epoch_g / len(loader)
    mean_c = epoch_c / len(loader)
    print(f'Epoch {epoch}: mean critic loss={mean_c:.4f}, mean generator loss={mean_g:.4f}')

    # Keep one rolling checkpoint pair so output storage remains small.
    generator_to_save = netG.module if isinstance(netG, nn.DataParallel) else netG
    critic_to_save = netC.module if isinstance(netC, nn.DataParallel) else netC
    torch.save(generator_to_save.state_dict(), OUTPUT_DIR / 'checkpoints' / 'generator.pth')
    torch.save(critic_to_save.state_dict(), OUTPUT_DIR / 'checkpoints' / 'critic.pth')
    if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
        print('Saved samples:', save_sample_grid(epoch))

plt.figure(figsize=(10, 4))
plt.plot(g_losses, label='Generator')
plt.plot(c_losses, label='Critic')
plt.xlabel('Training batch')
plt.ylabel('Loss')
plt.title('STL-10 WGAN-GP training losses')
plt.legend()
plt.tight_layout()
loss_plot = OUTPUT_DIR / 'training_metrics.png'
plt.savefig(loss_plot, dpi=140)
plt.show()
print('Training finished. Results are in:', OUTPUT_DIR)

In [ ]:
sample_files = sorted((OUTPUT_DIR / 'samples').glob('epoch_*.png'))
if sample_files:
    latest = sample_files[-1]
    print('Latest sample grid:', latest)
    plt.figure(figsize=(8, 8))
    plt.imshow(plt.imread(latest))
    plt.axis('off')
    plt.show()
print('Generator checkpoint:', OUTPUT_DIR / 'checkpoints' / 'generator.pth')
print('Critic checkpoint:   ', OUTPUT_DIR / 'checkpoints' / 'critic.pth')
print('Loss plot:           ', OUTPUT_DIR / 'training_metrics.png')